In [ ]:
# Atualiza dados
from app.services.pipeline import coletar_dados

from time import perf_counter

username = "felipe.cruz"
password = "#Gladoscruz.9851"
analise = "compra por necessidade"

dfs = coletar_dados(username, password, analise)

import pickle

with open("snapshot_dfs.pkl", "wb") as f:
	pickle.dump(dfs, f)

print("Snapshot salvo.")


In [ ]:
#Carerga Snapshot

import pickle

dfs = {}
with open("snapshot_dfs.pkl", "rb") as f:
	dfs = pickle.load(f)
print("Carregado com sucesso")

Carregado com sucesso


In [ ]:
import pandas as pd
import networkx as nx

print(f"{dfs.keys()}\n")


def sanitizar_dataframe(df, limite=0.8):
	df = df.copy()

	for col in df.columns:
		serie = df[col].astype(str).str.strip()

		tentativa_data = pd.to_datetime(
			serie, errors="coerce", dayfirst=True, format="%d/%m/%Y"
		)
		if tentativa_data.notna().mean() > limite:
			df[col] = tentativa_data
			continue

		serie_num = serie.str.replace(".", "", regex=False).str.replace(
			",", ".", regex=False
		)
		tentativa_num = pd.to_numeric(serie_num, errors="coerce")
		if tentativa_num.notna().mean() > limite:
			df[col] = tentativa_num
			continue

		df[col] = serie.replace({"": None})

	return df


def calc_data(dfs):

	## Ajuste Ordens ##
	ordens = sanitizar_dataframe(dfs.get("ordens"))
	ordens = ordens[
		[
			"Cliente",
			"Fábrica",
			"Ordem",
			"Pedido",
			"Item",
			"Saldo",
			"Representante",
			"Entrega Pedido",
			"Data Abertura",
		]
	]
	ordens = ordens.rename(columns={"Ordem": "Ordem Prod", "Saldo": "Saldo Prod"})

	colunas = ["Entrega Pedido", "Data Abertura"]
	for col in colunas:
		# Remove o ponto e garante que a coluna seja tratada como string
		ordens[col] = (
			ordens[col]
			.astype(str)
			.str.strip()
			.str.replace(r"[^\d]", "", regex=True)  # remove tudo que não for número
			.pipe(lambda s: pd.to_datetime(s, format="%d%m%Y", errors="coerce"))
		)
	## Ajuste Consumo ##
	consumo = sanitizar_dataframe(dfs.get("cons"))

	consumo["Item"] = consumo["Item"].str.split("-").str[0].str.strip()
	print(consumo.columns)
	consumo = consumo[["Item", "Baixa", "Consumo", "Local Prod.", "OP", "Familia"]]
	consumo = consumo.rename(columns={"OP": "Ordem Cons"})

	######## Item Pai ##########

	consumo["item_pai"] = consumo["Ordem Cons"].map(
		ordens.set_index("Ordem Prod")["Item"]
	)
	######## Importa dados ##########
	consumo = consumo.merge(ordens[["Item", "Ordem Prod"]], on="Item", how="left")

	consumo = consumo.merge(
		ordens[
			[
				"Cliente",
				"Fábrica",
				"Ordem Prod",
				"Pedido",
				"Saldo Prod",
				"Representante",
				"Entrega Pedido",
				"Data Abertura",
			]
		].rename(columns={"Ordem Prod": "Ordem", "Saldo Prod": "Saldo"}),
		left_on="Ordem Cons",
		right_on="Ordem",
		how="left",
	)

	######## Ordena as colunas ##########

	consumo = consumo[
		[
			"Ordem Prod",
			"Item",
			"Consumo",
			"Ordem Cons",
			"item_pai",
			"Saldo",
			"Pedido",
			"Representante",
			"Entrega Pedido",
			"Local Prod.",
			"Familia",
			"Cliente",
			"Fábrica",
			"Ordem",
			"Data Abertura",
			"Baixa",
		]
	]
	######## Calculos Baseados em estoque ##########
	estoque = sanitizar_dataframe(dfs.get("estoque"))
	consumo["estoque"] = (
		consumo["Item"].map(estoque.groupby("Item")["Qtde."].sum()).fillna(0)
	)

	CAMPOS_RAIZ = [
		"Cliente",
		"Fábrica",
		"Pedido",
		"Representante",
		"Entrega Pedido",
		"Data Abertura",
		"Saldo",
		"Baixa",
	]

	itens_existentes = set(consumo["Item"].unique())
	mapa = consumo.drop_duplicates("Ordem Cons").set_index("Ordem Cons")
	mapa_item_ordem = consumo.drop_duplicates("Item").set_index("Item")["Ordem Cons"]

	raiz_rows = []
	for ordem_cons in consumo["Ordem Cons"]:
		visitados = set()
		atual = ordem_cons
		resultado = None

		while atual in mapa.index:
			if atual in visitados:
				break
			visitados.add(atual)
			linha = mapa.loc[atual]
			item_pai = linha["item_pai"]

			if pd.isna(item_pai) or item_pai not in itens_existentes:
				resultado = linha[CAMPOS_RAIZ]
				break

			if item_pai not in mapa_item_ordem.index:
				resultado = linha[CAMPOS_RAIZ]
				break

			atual = mapa_item_ordem[item_pai]

		raiz_rows.append(
			resultado.to_dict()
			if resultado is not None
			else {c: None for c in CAMPOS_RAIZ}
		)

	raiz_df = pd.DataFrame(raiz_rows, index=consumo.index)
	raiz_df.columns = [f"raiz_{c}" for c in raiz_df.columns]
	consumo = pd.concat([consumo, raiz_df], axis=1)

	csv_path = "CSV/"
	consumo.to_excel(csv_path + "consumo.xlsx", index=False)


calc_data(dfs)

In [ ]:
# Cálculo com grafos
import pandas as pd
import networkx as nx

print(f"{dfs.keys()}\n")


def sanitizar_dataframe(df, limite=0.8):
	df = df.copy()

	for col in df.columns:
		serie = df[col].astype(str).str.strip()

		tentativa_data = pd.to_datetime(
			serie, errors="coerce", dayfirst=True, format="%d/%m/%Y"
		)
		if tentativa_data.notna().mean() > limite:
			df[col] = tentativa_data
			continue

		serie_num = serie.str.replace(".", "", regex=False).str.replace(
			",", ".", regex=False
		)
		tentativa_num = pd.to_numeric(serie_num, errors="coerce")
		if tentativa_num.notna().mean() > limite:
			df[col] = tentativa_num
			continue

		df[col] = serie.replace({"": None})

	return df


def calc_data(dfs):

	## Ajuste Ordens ##
	ordens = sanitizar_dataframe(dfs.get("ordens"))
	ordens = ordens[
		[
			"Cliente",
			"Fábrica",
			"Ordem",
			"Pedido",
			"Item",
			"Saldo",
			"Representante",
			"Entrega Pedido",
			"Data Abertura",
		]
	]
	ordens = ordens.rename(columns={"Ordem": "Ordem Prod", "Saldo": "Saldo Prod"})

	colunas = ["Entrega Pedido", "Data Abertura"]
	for col in colunas:
		# Remove o ponto e garante que a coluna seja tratada como string
		ordens[col] = (
			ordens[col]
			.astype(str)
			.str.strip()
			.str.replace(r"[^\d]", "", regex=True)  # remove tudo que não for número
			.pipe(lambda s: pd.to_datetime(s, format="%d%m%Y", errors="coerce"))
		)
	## Ajuste Consumo ##
	consumo = sanitizar_dataframe(dfs.get("cons"))
	consumo["Item"] = consumo["Item"].str.split("-").str[0].str.strip()
	consumo = consumo[
		["Tipo", "Item", "Baixa", "Consumo", "Local Prod.", "OP", "Familia", "Den. Item"]
	]
	consumo = consumo.rename(columns={"OP": "Ordem Cons"})

	######## Item Pai ##########
	consumo["item_pai"] = consumo["Ordem Cons"].map(
		ordens.set_index("Ordem Prod")["Item"]
	)

	######## Merge 1: traz Pedido/Cliente/etc via Ordem Cons (sem explosão) ##########
	consumo = consumo.merge(
		ordens[
			[
				"Cliente",
				"Fábrica",
				"Ordem Prod",
				"Pedido",
				"Saldo Prod",
				"Representante",
				"Entrega Pedido",
				"Data Abertura",
			]
		].rename(columns={"Ordem Prod": "Ordem", "Saldo Prod": "Saldo"}),
		left_on="Ordem Cons",
		right_on="Ordem",
		how="left",
	)

	######## Merge 2: Ordem Prod do componente via (Item, Pedido) ##########
	# Pedido já disponível — discrimina qual das múltiplas ordens do mesmo Item é a correta
	ordens_op = ordens[["Item", "Ordem Prod", "Pedido"]].copy()

	# Casos com Pedido > 0: usa (Item, Pedido) como chave — sem explosão
	consumo_com_pedido = consumo[consumo["Pedido"] > 0]
	consumo_sem_pedido = consumo[consumo["Pedido"] == 0]

	consumo_com_pedido = consumo_com_pedido.merge(
		ordens_op[ordens_op["Pedido"] > 0],
		on=["Item", "Pedido"],
		how="left",
	)

	# Casos Pedido=0: usa só Item, pega primeiro match (166 casos ambíguos inevitáveis)
	consumo_sem_pedido = consumo_sem_pedido.merge(
		ordens_op.drop_duplicates("Item")[["Item", "Ordem Prod"]],
		on="Item",
		how="left",
	)

	consumo = pd.concat([consumo_com_pedido, consumo_sem_pedido]).sort_index()

	######## Ordena as colunas ##########
	consumo = consumo[
		[
			"Tipo",
			"Ordem Prod",
			"Item",
			"Consumo",
			"Ordem Cons",
			"item_pai",
			"Saldo",
			"Pedido",
			"Representante",
			"Entrega Pedido",
			"Local Prod.",
			"Familia",
			"Cliente",
			"Fábrica",
			"Ordem",
			"Data Abertura",
			"Baixa",
			"Den. Item",
		]
	]
	######## Calculos Baseados em estoque ##########
	estoque = sanitizar_dataframe(dfs.get("estoque"))
	consumo["estoque"] = (
		consumo["Item"].map(estoque.groupby("Item")["Qtde."].sum()).fillna(0)
	)

	from tqdm import tqdm
	import networkx as nx

	######## Propagação dos atributos da raiz via NetworkX ##########

	######## Propagação dos atributos da raiz ##########

	CAMPOS_RAIZ = [
		"Item",
		"Cliente",
		"Fábrica",
		"Pedido",
		"Representante",
		"Entrega Pedido",
		"Data Abertura",
		"Saldo",
		"Baixa",
	]

	itens_existentes = set(consumo["Item"].unique())
	raizes = set(consumo["item_pai"].dropna()) - itens_existentes

	# Nível 0: ordens cujo item_pai é raiz real
	cache_raiz = {}
	for _, row in (
		consumo[consumo["item_pai"].isin(raizes)]
		.drop_duplicates("Ordem Cons")
		.iterrows()
	):
		cache_raiz[row["Ordem Cons"]] = row[CAMPOS_RAIZ].to_dict()

	# Mapa Ordem Prod → [(Ordem Cons consumidora, Pedido_Ordem, Pedido_Contexto)]
	# Construído direto de ordens x consumo — não depende de Ordem Prod em consumo
	mapa_raw = (
		ordens[["Ordem Prod", "Item", "Pedido"]]
		.rename(columns={"Pedido": "Pedido_Ordem"})
		.merge(
			consumo[["Item", "Ordem Cons", "Pedido"]].drop_duplicates(
				["Item", "Ordem Cons"]
			),
			on="Item",
		)
	)

	# Agrupa: Ordem Prod → lista de (Ordem Cons, Pedido_Ordem, Pedido_Contexto)
	mapa_op_para_pais = {}
	for _, row in mapa_raw.iterrows():
		op = row["Ordem Prod"]
		if op not in mapa_op_para_pais:
			mapa_op_para_pais[op] = []
		mapa_op_para_pais[op].append(
			(row["Ordem Cons"], row["Pedido_Ordem"], row["Pedido"])
		)

	def achar_pai_no_cache(ordem_cons_filho, pedido_filho):
		pais = mapa_op_para_pais.get(ordem_cons_filho, [])
		pais_no_cache = [(oc, po, pc) for oc, po, pc in pais if oc in cache_raiz]
		if not pais_no_cache:
			return None
		if len(pais_no_cache) == 1:
			return cache_raiz[pais_no_cache[0][0]]
		# Ambíguo: Pedido_Ordem == Pedido_Contexto é o match correto
		match = [(oc, po, pc) for oc, po, pc in pais_no_cache if po == pc]
		if len(match) == 1:
			return cache_raiz[match[0][0]]
		# Fallback: Pedido do filho bate com Pedido do contexto
		match2 = [(oc, po, pc) for oc, po, pc in pais_no_cache if pc == pedido_filho]
		if match2:
			return cache_raiz[match2[0][0]]
		return cache_raiz[pais_no_cache[0][0]]

	# BFS top-down
	for nivel in range(1, 10):
		pendentes = consumo[~consumo["Ordem Cons"].isin(cache_raiz)].drop_duplicates(
			"Ordem Cons"
		)
		novos = 0
		for _, row in pendentes.iterrows():
			resultado = achar_pai_no_cache(row["Ordem Cons"], row["Pedido"])
			if resultado:
				cache_raiz[row["Ordem Cons"]] = resultado
				novos += 1
		print(f"Nível {nivel}: {novos} novas entradas resolvidas")
		if novos == 0:
			break

	raiz_df = (
		consumo["Ordem Cons"]
		.map(cache_raiz)
		.apply(lambda x: x if isinstance(x, dict) else {c: None for c in CAMPOS_RAIZ})
	)
	raiz_df = pd.DataFrame(raiz_df.tolist(), index=consumo.index)
	raiz_df.columns = [f"raiz_{c}" for c in CAMPOS_RAIZ]
	consumo = pd.concat([consumo, raiz_df], axis=1)

	#### Gera Excel ####
	print("gerar excel")
	csv_path = "CSV/"
	consumo.to_excel(csv_path + "consumo.xlsx", index=False)
	print("Concluido")


calc_data(dfs)

dict_keys(['apoio_compras', 'ordens', 'cons', 'estoque'])

Nível 1: 5339 novas entradas resolvidas
Nível 2: 1379 novas entradas resolvidas
Nível 3: 15 novas entradas resolvidas
Nível 4: 0 novas entradas resolvidas
gerar excel
Concluido


In [11]:
print(dfs.get("cons").columns)

Index(['Tipo', 'Grupo', 'Item', 'Local Estoque', 'Baixa', 'Situação',
       'Den. Item', 'Consumo', 'Local Prod.', 'OP', 'Familia', 'Família'],
      dtype='str')
